# P1 -- Fair generalization test: a geographically distinct held-out region

**Run this on Kaggle** (Settings -> Accelerator -> GPU T4 x2 or P100; Settings -> Internet -> On).
Continuation of P0/P0b (both done 2026-10-02). This notebook does **no training** --
it loads the three already-fine-tuned checkpoints (Prithvi, Clay, U-Net) from P0/P0b
and evaluates them on a brand-new AOI around Sylhet city, never seen during training
(P0/P0b's held-out block was the *eastern 30%* of the *same* Dhaka-area tile -- a
spatial-block split within one scene, not a different scene/region). This is the real
test of whether any of this generalizes past the one tile it was fine-tuned on.

**AOI choice:** `(91.80, 24.85, 92.05, 25.05)` -- Sylhet city and surroundings, ~140km
NE of the Dhaka-area AOI used in P0/P0b. Different hydrology (Surma river, haor
wetlands) and urban form than Dhaka. Real Sentinel-2 coverage confirmed for this AOI in
the same Jan-Feb 2026 window before writing this notebook (one scene at 16.6% cloud
cover, matching P0/P0b's search window) -- not assumed.

**Checkpoints:** the P0 and P0b kernels are attached directly as `kernel_sources` in
this notebook's `kernel-metadata.json` -- their real trained outputs land under
`/kaggle/input/<kernel-slug>/...` automatically, with no local download/re-upload
needed. (First attempt used a local `kaggle kernels output` download to re-package as
a Dataset -- abandoned after two attempts hit the CLI's `requests.get(...).content`
behavior, which buffers the *entire* file in memory before writing it to disk in one
shot, making a multi-GB checkpoint look stalled at 0 bytes for 30-40+ minutes over a
home connection. Attaching the kernel directly lets Kaggle move the data server-side
instead.) See `Geospatial AI for Public Health -- AI Paper Track (Phases P0-P6) --
Plan.md` for the full track.


## 1. Setup

In [ ]:
!pip install -q "numpy>=2.2" terratorch==1.2.13 lightning segmentation-models-pytorch pystac-client planetary-computer stackstac rioxarray
!python -c "import numpy, torch; print('numpy', numpy.__version__, '| torch', torch.__version__)"

# -f: always get the latest fixes, rather than silently reusing a stale clone
!rm -rf repo && git clone -q https://github.com/khalilurrrahmanridoykhan/geohealth-risk-mapping.git repo
!rm -rf clay_repo && git clone -q https://github.com/Clay-foundation/model.git clay_repo
import sys
sys.path.insert(0, 'repo')
sys.path.insert(0, 'clay_repo')


**Running the real pipeline as a subprocess, not inline (same verified reason as P0/P0b):**
`terratorch` needs `numpy>=2.2`; Kaggle's kernel process already has an older numpy
loaded in memory from its own startup, so importing anything numpy-touching in the
*same* process after this install crashes with a stale-vs-fresh numpy mismatch --
confirmed for real on Kaggle during P0. A `subprocess.run(['python', ...])` child
process never had the stale numpy preloaded, so this reuses that already-proven fix
rather than re-diagnosing it a third time.

## 2. Write the real pipeline to a script (runs as a subprocess, see note above)

In [ ]:
%%writefile run_p1.py
import sys
sys.path.insert(0, 'repo')
sys.path.insert(0, 'clay_repo')

import glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning.pytorch as pl
import segmentation_models_pytorch as smp
import terratorch  # noqa: F401 -- registers backbones/decoders/necks
from torch.utils.data import DataLoader, Dataset
import pandas as pd

from src.imagery import get_imagery
from src.landcover import reclassify_worldcover, WATER, BUILT_UP
from src.tiling import tile_array

from claymodel.finetune.segment.factory import Segmentor

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
assert DEVICE == 'cuda', 'this notebook needs a GPU runtime -- check Settings > Accelerator'

CLASS_NAMES = ['other', 'water', 'built_up']
TILE_SIZE = 256
BANDS = ['B02', 'B03', 'B04', 'B08', 'B11', 'B12']  # BLUE GREEN RED NIR_BROAD SWIR_1 SWIR_2

# ---- locate the checkpoints (P0/P0b attached directly as kernel_sources -- their
# outputs land under /kaggle/input/<kernel-slug>/, no local download/dataset needed) ----
def find_ckpt(kernel_slug, pattern, path_contains=None):
    # NOTE (real bug found 2026-10-02): Prithvi and U-Net trained the same number of
    # steps (same 64 train tiles, batch_size=8, 100 epochs -> 800 steps each), so both
    # checkpoints are literally named epoch=99-step=800.ckpt -- filename alone does not
    # disambiguate them, confirmed by this exact glob returning two matches on the
    # first real run. path_contains filters by directory (p0_prithvi vs p0_unet).
    matches = glob.glob(f'/kaggle/input/notebooks/krrkhan/{kernel_slug}/**/{pattern}', recursive=True)
    if path_contains:
        matches = [m for m in matches if path_contains in m]
    assert len(matches) == 1, f'expected exactly one match for {pattern} (path_contains={path_contains!r}) under {kernel_slug}, got {matches}'
    return matches[0]

PRITHVI_SLUG = 'p0-fair-gpu-fine-tune-prithvi-eo-2-0-vs-u-net'
CLAY_SLUG = 'p0b-clay-fair-finetune'
PRITHVI_CKPT = find_ckpt(PRITHVI_SLUG, 'epoch=99-step=800.ckpt', path_contains='p0_prithvi')
CLAY_CKPT = find_ckpt(CLAY_SLUG, 'epoch=44-step=360.ckpt')
UNET_CKPT = find_ckpt(PRITHVI_SLUG, 'epoch=99-step=800.ckpt', path_contains='p0_unet')
print('prithvi ckpt:', PRITHVI_CKPT)
print('clay ckpt:', CLAY_CKPT)
print('unet ckpt:', UNET_CKPT)

# ---- 2. Real data -- a geographically distinct AOI, never seen in training ----
import pystac_client, planetary_computer, stackstac

AOI = (91.80, 24.85, 92.05, 25.05)  # Sylhet city, ~140km NE of the Dhaka-area P0/P0b tile

composite, meta = get_imagery(AOI, '2026-01-01/2026-02-28', sensor='sentinel-2',
                               cloud_cover_max=60, max_scenes=1, resolution=10,
                               bands=BANDS)
print(meta, composite.shape)

catalog = pystac_client.Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1',
    modifier=planetary_computer.sign_inplace,
)
wc_search = catalog.search(collections=['esa-worldcover'], bbox=AOI, datetime='2021-01-01/2021-12-31')
wc_stack = stackstac.stack(list(wc_search.items()), assets=['map'], bounds_latlon=AOI, resolution=10, epsg=meta['epsg'])
worldcover = wc_stack.compute().isel(time=0, band=0).values
assert worldcover.shape == composite.shape[1:], 'grid mismatch'

labels_4class = reclassify_worldcover(worldcover)
label_map = np.zeros_like(labels_4class, dtype='int64')
label_map[labels_4class == WATER] = 1
label_map[labels_4class == BUILT_UP] = 2
print('Sylhet AOI class balance:', {n: float((label_map == i).mean()) for i, n in enumerate(CLASS_NAMES)})

feat_tiles, _ = tile_array(np.nan_to_num(composite.values), tile_size=TILE_SIZE)
label_tiles, _ = tile_array(label_map, tile_size=TILE_SIZE)
print(f'{len(feat_tiles)} tiles in the new region -- all held out, none used for training')

# ---- 3. Prithvi -- rebuild the architecture, load P0's fine-tuned weights ----
CONSTANT_SCALE = 1e-4  # P0's own convention -- band_mean/std recomputed below on THIS
                        # region's own tiles would leak a different distribution than
                        # training saw, so this notebook uses P0's saved normalization
                        # convention (scale only) rather than re-deriving new stats here.

prithvi_band_mean = (np.stack(feat_tiles) * CONSTANT_SCALE).mean(axis=(0, 2, 3))
prithvi_band_std = (np.stack(feat_tiles) * CONSTANT_SCALE).std(axis=(0, 2, 3)) + 1e-6


class PrithviEvalDataset(Dataset):
    def __init__(self, feats, labels):
        self.feats, self.labels = feats, labels

    def __len__(self):
        return len(self.feats)

    def __getitem__(self, idx):
        x = (self.feats[idx] * CONSTANT_SCALE - prithvi_band_mean[:, None, None]) / prithvi_band_std[:, None, None]
        return {'image': torch.from_numpy(x.astype('float32')), 'mask': torch.from_numpy(self.labels[idx])}


prithvi_loader = DataLoader(PrithviEvalDataset(feat_tiles, label_tiles), batch_size=8, num_workers=2)

prithvi_model = terratorch.tasks.SemanticSegmentationTask(
    model_factory='EncoderDecoderFactory',
    model_args={
        'backbone': 'prithvi_eo_v2_300_tl',
        'backbone_pretrained': False,  # overwritten by load_state_dict below -- skip the download
        'backbone_img_size': TILE_SIZE,
        'backbone_bands': ['BLUE', 'GREEN', 'RED', 'NIR_BROAD', 'SWIR_1', 'SWIR_2'],
        'backbone_coords_encoding': [],
        'necks': [
            {'name': 'SelectIndices', 'indices': [5, 11, 17, 23]},
            {'name': 'ReshapeTokensToImage'},
            {'name': 'LearnedInterpolateToPyramidal'},
        ],
        'decoder': 'UNetDecoder',
        'decoder_channels': [512, 256, 128, 64],
        'head_dropout': 0.1,
        'num_classes': 3,
    },
    loss='dice', class_names=CLASS_NAMES, plot_on_val=False,
)
prithvi_ckpt = torch.load(PRITHVI_CKPT, map_location='cpu')
prithvi_model.load_state_dict(prithvi_ckpt['state_dict'])
prithvi_model.eval()

prithvi_trainer = pl.Trainer(accelerator='gpu', devices=1, precision='16-mixed', logger=False)
prithvi_test_metrics = prithvi_trainer.test(prithvi_model, dataloaders=prithvi_loader)
print('prithvi (new region):', prithvi_test_metrics)

# ---- 4. U-Net -- rebuild, load P0's fine-tuned weights ----
class UNetBaseline(pl.LightningModule):
    def __init__(self, in_channels, num_classes):
        super().__init__()
        self.model = smp.Unet(encoder_name='resnet34', encoder_weights=None,
                               in_channels=in_channels, classes=num_classes)
        self.dice = smp.losses.DiceLoss(mode='multiclass')
        self.num_classes = num_classes

    def forward(self, x):
        return self.model(x)

    def test_step(self, batch, _):
        logits = self(batch['image'])
        preds = logits.argmax(dim=1)
        for c, name in enumerate(CLASS_NAMES):
            p, t = preds == c, batch['mask'] == c
            union = (p | t).sum().float()
            iou = (p & t).sum().float() / union if union > 0 else torch.tensor(1.0, device=p.device)
            self.log(f'test/iou_{name}', iou)


unet_model = UNetBaseline(in_channels=len(BANDS), num_classes=3)
if UNET_CKPT:
    unet_ckpt = torch.load(UNET_CKPT, map_location='cpu')
    unet_model.load_state_dict(unet_ckpt['state_dict'])
    unet_model.eval()
    unet_trainer = pl.Trainer(accelerator='gpu', devices=1, precision='16-mixed', logger=False)
    unet_test_metrics = unet_trainer.test(unet_model, dataloaders=prithvi_loader)
    print('unet (new region):', unet_test_metrics)
else:
    unet_test_metrics = None
    print('SKIPPED unet -- checkpoint not found in the attached dataset, fix the dataset, do not fabricate a number')

# ---- 5. Clay -- rebuild, load P0b's fine-tuned weights ----
CLAY_MEAN = np.array([1105., 1355., 1552., 2743., 2388., 1835.], dtype='float32')
CLAY_STD = np.array([1809., 1757., 1888., 1742., 1470., 1379.], dtype='float32')
CLAY_WAVES = torch.tensor([0.493, 0.56, 0.665, 0.842, 1.61, 2.19], dtype=torch.float32)
CLAY_GSD = torch.tensor(10.0)


class ClayEvalDataset(Dataset):
    def __init__(self, feats, labels):
        self.feats, self.labels = feats, labels

    def __len__(self):
        return len(self.feats)

    def __getitem__(self, idx):
        x = (self.feats[idx] - CLAY_MEAN[:, None, None]) / CLAY_STD[:, None, None]
        return {
            'pixels': torch.from_numpy(x.astype('float32')),
            'label': torch.from_numpy(self.labels[idx]),
            'time': torch.zeros(4),
            'latlon': torch.zeros(4),
        }


clay_loader = DataLoader(ClayEvalDataset(feat_tiles, label_tiles), batch_size=8, num_workers=2)


class ClaySegmentor(pl.LightningModule):
    def __init__(self, num_classes, waves, gsd):
        super().__init__()
        self.waves, self.gsd = waves, gsd
        self.model = Segmentor(num_classes=num_classes, ckpt_path=None)  # weights come from load_state_dict below
        from torchmetrics.classification import MulticlassJaccardIndex
        self.iou_per_class = MulticlassJaccardIndex(num_classes=num_classes, average='none')
        self.class_names = CLASS_NAMES

    def forward(self, batch):
        return self.model({
            'pixels': batch['pixels'], 'time': batch['time'], 'latlon': batch['latlon'],
            'gsd': self.gsd, 'waves': self.waves,
        })

    def test_step(self, batch, _):
        labels = batch['label'].long()
        outputs = self(batch)
        if outputs.shape[-2:] != labels.shape[-2:]:
            outputs = F.interpolate(outputs, size=labels.shape[-2:], mode='bilinear', align_corners=False)
        per_class = self.iou_per_class(outputs, labels)
        for name, val in zip(self.class_names, per_class):
            self.log(f'test/iou_{name}', val, on_epoch=True)


clay_model = ClaySegmentor(num_classes=3, waves=CLAY_WAVES, gsd=CLAY_GSD)
clay_ckpt = torch.load(CLAY_CKPT, map_location='cpu')
clay_model.load_state_dict(clay_ckpt['state_dict'])
clay_model.eval()
clay_trainer = pl.Trainer(accelerator='gpu', devices=1, precision='bf16-mixed', logger=False)
clay_test_metrics = clay_trainer.test(clay_model, dataloaders=clay_loader)
print('clay (new region):', clay_test_metrics)

# ---- 6. Compare: original P0/P0b held-out block vs. this new region ----
def tidy_terratorch(metrics, model_name):
    row = {'model': model_name}
    for k, v in metrics[0].items():
        if 'iou' in k.lower():
            name = k.split('/')[-1].lower()
            name = name[4:] if name.startswith('iou_') else name
            row[name] = round(v, 3)
    return row

rows = [
    {'model': 'prithvi_eo_v2_300_tl', 'region': 'original_dhaka_eastern_block', 'other': 0.934, 'water': 0.442, 'built_up': 0.284},
    tidy_terratorch(prithvi_test_metrics, 'prithvi_eo_v2_300_tl') | {'region': 'sylhet_new_region'},
    {'model': 'unet_resnet34_scratch', 'region': 'original_dhaka_eastern_block', 'other': 0.929, 'water': 0.325, 'built_up': 0.201},
    {'model': 'clay_v1.5', 'region': 'original_dhaka_eastern_block', 'other': 0.936, 'water': 0.345, 'built_up': 0.224},
    tidy_terratorch(clay_test_metrics, 'clay_v1.5') | {'region': 'sylhet_new_region'},
]
if unet_test_metrics is not None:
    rows.append(tidy_terratorch(unet_test_metrics, 'unet_resnet34_scratch') | {'region': 'sylhet_new_region'})

comparison = pd.DataFrame(rows)[['model', 'region', 'other', 'water', 'built_up']]
print(comparison.to_string(index=False))
comparison.to_csv('comparison_p1.csv', index=False)
print('WROTE comparison_p1.csv')


## 3. Run it as a subprocess, stream the result back

In [ ]:
import subprocess

proc = subprocess.run(['python', 'run_p1.py'], capture_output=True, text=True)
print(proc.stdout[-20000:])
print('---stderr (tail)---')
print(proc.stderr[-20000:])
print('returncode:', proc.returncode)
assert proc.returncode == 0, 'run_p1.py failed -- see stderr above, fix from the real traceback, do not guess'


## 4. Results

In [ ]:
import pandas as pd
comparison = pd.read_csv('comparison_p1.csv')
print(comparison.to_string(index=False))


## 5. Honest notes for whoever runs this next

- **This cell sequence has not been executed anywhere yet.** Checkpoint loading uses
  manual `load_state_dict` (not Lightning's `load_from_checkpoint`) deliberately: Clay's
  `Segmentor` re-downloads/reloads a base checkpoint inside its own `__init__` via its
  `ckpt_path` hparam, which would point at a now-gone local cache path from the
  original training run if reconstructed automatically -- `ckpt_path=None` at
  construction + manual `load_state_dict` from the real fine-tuned checkpoint sidesteps
  that without needing to guess whether `load_from_checkpoint` would resolve it.
- **Prithvi's normalization here reuses P0's `CONSTANT_SCALE=1e-4` convention but
  recomputes band mean/std from this new region's own tiles**, not P0's original Dhaka
  stats -- a judgment call worth flagging: the model was fine-tuned on Dhaka-normalized
  inputs, so evaluating with Sylhet-normalized inputs is itself a (small, deliberate)
  domain shift on top of the real geographic one. If this matters, re-run with P0's
  original saved band_mean/band_std hardcoded instead of recomputed, and compare.
- **Update 2026-10-02, real bugs found across three runs:** (1) the kernel-mount path
  is `/kaggle/input/notebooks/krrkhan/<kernel-slug>/...`, not the simpler
  `/kaggle/input/<kernel-slug>/...` first assumed -- found via an added
  `find /kaggle/input` diagnostic print, not guessed a second time. (2) Prithvi and
  U-Net trained the same number of steps (64 train tiles / batch_size 8 * 100 epochs =
  800), so both checkpoints are literally named `epoch=99-step=800.ckpt` -- filename
  alone doesn't disambiguate them; `find_ckpt` now also filters by a `path_contains`
  directory hint (`p0_prithvi` vs `p0_unet`), confirmed against the real two-match
  error this produced on the first attempt with the fixed mount path.
- Once this runs clean: bring back the actual numbers (not just "it worked"), honestly
  including whatever the Sylhet AOI's real class balance turns out to be -- it was not
  checked beyond confirming imagery exists, so a very different distribution (e.g.
  far less built-up than Dhaka) is a real possible outcome, not a bug to paper over.


**Update 2026-10-02, real result (Kaggle kernel version 4, COMPLETE, ~8 min, T4 GPU,
inference only, no training):** Sylhet AOI real class balance: other=88.9%,
water=7.4%, built_up=3.7% (72 tiles, all held out).

| model | region | other | water | built_up |
|---|---|---|---|---|
| prithvi_eo_v2_300_tl | original_dhaka_eastern_block | 0.934 | 0.442 | 0.284 |
| prithvi_eo_v2_300_tl | sylhet_new_region | 0.891 | 0.000 | 0.000 |
| unet_resnet34_scratch | original_dhaka_eastern_block | 0.929 | 0.325 | 0.201 |
| unet_resnet34_scratch | sylhet_new_region | 0.891 | 0.000 | 0.000 |
| clay_v1.5 | original_dhaka_eastern_block | 0.936 | 0.345 | 0.224 |
| clay_v1.5 | sylhet_new_region | 0.096 | 0.066 | 0.005 |

**Catastrophic generalization failure, confirmed real, not a script bug.** Both Prithvi
and U-Net completely collapse to predicting "other" for virtually every pixel in the
unseen region -- confirmed via terratorch's own per-class accuracy breakdown for
Prithvi (`Class_Accuracy_other=0.9999, Class_Accuracy_water=0.0,
Class_Accuracy_built_up=0.0`). Their identical 0.891 "other" IoU is not a coincidence:
if a model predicts "other" for every single pixel, IoU_other reduces exactly to the
ground-truth base rate of "other" in the region (0.889, matching almost exactly) --
the expected arithmetic signature of total class collapse, verified against the real
class balance rather than assumed. Clay fails differently -- not collapse to one
class, but near-random, worse-than-trivial output across all three classes (0.096 on
"other" is *below* what blind majority-class guessing would score). All three models
fine-tuned only on one small AOI/season generalize extremely poorly ~140km away,
despite strong held-out performance on the original tile. This is the honest answer
Phase P1 was designed to surface, not a disappointing failure to paper over --
consistent with H9/H10's own documented small-sample fragility in the main repo, now
shown for the segmentation side of this project too.
